# 🔥 CNN Model - Comprehensive Stress Testing

This notebook performs extensive stress testing of the trained CNN model with multiple noise variations and SNR levels.

## Imports and Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import seaborn as sns
import pandas as pd
import torchaudio
import os
import random

from torchaudio.transforms import TimeMasking, FrequencyMasking
from collections import defaultdict
from torchvision.transforms import Compose, ToTensor
from torch.utils.data import DataLoader, Dataset
from sklearn import metrics
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

pd.set_option('future.no_silent_downcasting', True)

In [ ]:
# Load model paths and device
best_model_path = "model/CNN-Best.pkl"
model_path = "model/CNN-Final.pkl"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Load Noise Library and Functions from Main Notebook

In [ ]:
import sys, os
if '.' not in sys.path:
    sys.path.insert(0, '.')
from cnn_shared import NoiseLibrary

# Initialize global noise library
print("Initializing Noise Library...")
noise_library = NoiseLibrary()
print("✅ Noise Library Ready!\n")

In [ ]:
# calculate_rms, add_noise_snr, add_white_noise_snr, add_gaussian_noise_snr,
# and reduce_volume_db are all imported from cnn_shared via the NoiseLibrary cell above
from cnn_shared import (
    calculate_rms, add_noise_snr, add_white_noise_snr, add_gaussian_noise_snr, reduce_volume_db,
)
print("✅ Noise functions loaded from cnn_shared!")

## Load Dataset and Model

In [ ]:
from cnn_shared import AudioDataset

# Load dataset
sample_rate = 44100
to_mel_spectrogram = torchaudio.transforms.MelSpectrogram(sample_rate, n_mels=64, hop_length=300, n_fft=2048, win_length=1024)
mel_spectrogram_to_numpy = lambda spectrogram: spectrogram.log2()[0,:,:].numpy()
transforms = Compose([to_mel_spectrogram, mel_spectrogram_to_numpy, ToTensor()])
dataset = AudioDataset('Data/segmented/alpha-large', transforms)
print("Number of classes:", dataset.num_classes())

In [ ]:
# Split dataset
targets = [data[1] for data in dataset]

train_indices, tmp_indices = train_test_split(
    range(len(dataset)), 
    test_size=0.3,
    stratify=targets
)

val_indices, test_indices = train_test_split(
    tmp_indices, 
    test_size=0.33,
    stratify=[targets[i] for i in tmp_indices]
)

init_test_set = torch.utils.data.Subset(dataset, test_indices)
print(f"Test set size: {len(init_test_set)}")

In [ ]:
import sys, os
if '.' not in sys.path:
    sys.path.insert(0, '.')
from cnn_shared import CNN, init_weights, get_device

model = CNN().to(device)
model.load_state_dict(torch.load(best_model_path))
model.eval()
print("✅ Model loaded successfully!")

In [ ]:
# Create label dictionary
digits = [str(digit) for digit in range(10)]
alphabet = [chr(ascii_code) for ascii_code in range(ord('A'), ord('Z') + 1)]
all_characters = digits + alphabet
label_dict = {i: all_characters[i] for i in range(len(all_characters))}
print("Label dictionary created:", len(label_dict), "classes")

## Stress Test Augmentation Classes

In [ ]:
class StressTestAugmentation:
    """
    Create stress test datasets with specific noise configurations
    """
    def __init__(self, noise_library, noise_type='clean', snr_db=None, 
                 volume_reduction_db=None, category=None):
        """
        Args:
            noise_library: NoiseLibrary instance
            noise_type: 'clean', 'real', 'white', 'gaussian', 'mixed'
            snr_db: SNR in dB (None for clean)
            volume_reduction_db: Volume reduction in dB (None for no reduction)
            category: Specific noise category for real noise ('city', 'office', 'classroom', 'common')
        """
        self.noise_library = noise_library
        self.noise_type = noise_type
        self.snr_db = snr_db
        self.volume_reduction_db = volume_reduction_db
        self.category = category
        
    def __call__(self, signal):
        """Apply specific noise augmentation"""
        if self.noise_type == 'clean':
            augmented = signal.clone()
        elif self.noise_type == 'real':
            signal_length = signal.shape[1] if signal.dim() > 1 else signal.shape[0]
            noise_segment = self.noise_library.get_noise_segment(signal_length, self.category)
            if noise_segment is not None:
                if noise_segment.shape[0] != signal.shape[0]:
                    noise_segment = noise_segment[:signal.shape[0], :]
                augmented = add_noise_snr(signal, noise_segment, self.snr_db)
            else:
                augmented = signal.clone()
        elif self.noise_type == 'white':
            augmented = add_white_noise_snr(signal, self.snr_db)
        elif self.noise_type == 'gaussian':
            augmented = add_gaussian_noise_snr(signal, self.snr_db)
        elif self.noise_type == 'mixed':
            # Apply multiple noise types together
            signal_length = signal.shape[1] if signal.dim() > 1 else signal.shape[0]
            
            # Add real noise
            noise_segment = self.noise_library.get_noise_segment(signal_length, self.category)
            if noise_segment is not None:
                if noise_segment.shape[0] != signal.shape[0]:
                    noise_segment = noise_segment[:signal.shape[0], :]
                augmented = add_noise_snr(signal, noise_segment, self.snr_db)
            else:
                augmented = signal.clone()
            
            # Add white noise on top
            augmented = add_white_noise_snr(augmented, self.snr_db + 3)  # Slightly higher SNR
        else:
            augmented = signal.clone()
        
        # Apply volume reduction if specified
        if self.volume_reduction_db is not None:
            augmented = reduce_volume_db(augmented, self.volume_reduction_db)
        
        return augmented

print("✅ Stress Test Augmentation Class Created!")

## Define Stress Test Configurations

In [ ]:
# Define comprehensive stress test configurations
# High SNR values: 15-30 dB (cleaner signals, more challenging for robustness)
stress_test_configs = [
    # 1. Clean baseline
    {'name': 'Clean (Baseline)', 'noise_type': 'clean', 'snr_db': None, 'vol_reduction': None, 'category': None},
    
    # 2-5. Real ambient noise - City (High SNR)
    {'name': 'City Noise - SNR 30dB', 'noise_type': 'real', 'snr_db': 30, 'vol_reduction': None, 'category': 'city'},
    {'name': 'City Noise - SNR 25dB', 'noise_type': 'real', 'snr_db': 25, 'vol_reduction': None, 'category': 'city'},
    {'name': 'City Noise - SNR 20dB', 'noise_type': 'real', 'snr_db': 20, 'vol_reduction': None, 'category': 'city'},
    {'name': 'City Noise - SNR 15dB', 'noise_type': 'real', 'snr_db': 15, 'vol_reduction': None, 'category': 'city'},
    
    # 6-9. Real ambient noise - Office (High SNR)
    {'name': 'Office Noise - SNR 30dB', 'noise_type': 'real', 'snr_db': 30, 'vol_reduction': None, 'category': 'office'},
    {'name': 'Office Noise - SNR 25dB', 'noise_type': 'real', 'snr_db': 25, 'vol_reduction': None, 'category': 'office'},
    {'name': 'Office Noise - SNR 20dB', 'noise_type': 'real', 'snr_db': 20, 'vol_reduction': None, 'category': 'office'},
    {'name': 'Office Noise - SNR 15dB', 'noise_type': 'real', 'snr_db': 15, 'vol_reduction': None, 'category': 'office'},
    
    # 10-13. Real ambient noise - Classroom (High SNR)
    {'name': 'Classroom Noise - SNR 30dB', 'noise_type': 'real', 'snr_db': 30, 'vol_reduction': None, 'category': 'classroom'},
    {'name': 'Classroom Noise - SNR 25dB', 'noise_type': 'real', 'snr_db': 25, 'vol_reduction': None, 'category': 'classroom'},
    {'name': 'Classroom Noise - SNR 20dB', 'noise_type': 'real', 'snr_db': 20, 'vol_reduction': None, 'category': 'classroom'},
    {'name': 'Classroom Noise - SNR 15dB', 'noise_type': 'real', 'snr_db': 15, 'vol_reduction': None, 'category': 'classroom'},
    
    # 14-17. Real ambient noise - Common (High SNR)
    {'name': 'Common Noise - SNR 30dB', 'noise_type': 'real', 'snr_db': 30, 'vol_reduction': None, 'category': 'common'},
    {'name': 'Common Noise - SNR 25dB', 'noise_type': 'real', 'snr_db': 25, 'vol_reduction': None, 'category': 'common'},
    {'name': 'Common Noise - SNR 20dB', 'noise_type': 'real', 'snr_db': 20, 'vol_reduction': None, 'category': 'common'},
    {'name': 'Common Noise - SNR 15dB', 'noise_type': 'real', 'snr_db': 15, 'vol_reduction': None, 'category': 'common'},
    
    # 18-21. White noise (High SNR)
    {'name': 'White Noise - SNR 30dB', 'noise_type': 'white', 'snr_db': 30, 'vol_reduction': None, 'category': None},
    {'name': 'White Noise - SNR 25dB', 'noise_type': 'white', 'snr_db': 25, 'vol_reduction': None, 'category': None},
    {'name': 'White Noise - SNR 20dB', 'noise_type': 'white', 'snr_db': 20, 'vol_reduction': None, 'category': None},
    {'name': 'White Noise - SNR 15dB', 'noise_type': 'white', 'snr_db': 15, 'vol_reduction': None, 'category': None},
    
    # 22-25. Gaussian noise (High SNR)
    {'name': 'Gaussian Noise - SNR 30dB', 'noise_type': 'gaussian', 'snr_db': 30, 'vol_reduction': None, 'category': None},
    {'name': 'Gaussian Noise - SNR 25dB', 'noise_type': 'gaussian', 'snr_db': 25, 'vol_reduction': None, 'category': None},
    {'name': 'Gaussian Noise - SNR 20dB', 'noise_type': 'gaussian', 'snr_db': 20, 'vol_reduction': None, 'category': None},
    {'name': 'Gaussian Noise - SNR 15dB', 'noise_type': 'gaussian', 'snr_db': 15, 'vol_reduction': None, 'category': None},
    
    # 26-29. Mixed noise (Real + White, High SNR)
    {'name': 'Mixed Noise (Real+White) - SNR 30dB', 'noise_type': 'mixed', 'snr_db': 30, 'vol_reduction': None, 'category': None},
    {'name': 'Mixed Noise (Real+White) - SNR 25dB', 'noise_type': 'mixed', 'snr_db': 25, 'vol_reduction': None, 'category': None},
    {'name': 'Mixed Noise (Real+White) - SNR 20dB', 'noise_type': 'mixed', 'snr_db': 20, 'vol_reduction': None, 'category': None},
    {'name': 'Mixed Noise (Real+White) - SNR 15dB', 'noise_type': 'mixed', 'snr_db': 15, 'vol_reduction': None, 'category': None},
    
    # 30-33. City noise with volume reduction (High SNR)
    {'name': 'City SNR 25dB + Vol -3dB', 'noise_type': 'real', 'snr_db': 25, 'vol_reduction': 3, 'category': 'city'},
    {'name': 'City SNR 25dB + Vol -6dB', 'noise_type': 'real', 'snr_db': 25, 'vol_reduction': 6, 'category': 'city'},
    {'name': 'City SNR 25dB + Vol -9dB', 'noise_type': 'real', 'snr_db': 25, 'vol_reduction': 9, 'category': 'city'},
    {'name': 'City SNR 25dB + Vol -12dB', 'noise_type': 'real', 'snr_db': 25, 'vol_reduction': 12, 'category': 'city'},
    
    # 34-37. Office noise with volume reduction (High SNR)
    {'name': 'Office SNR 25dB + Vol -3dB', 'noise_type': 'real', 'snr_db': 25, 'vol_reduction': 3, 'category': 'office'},
    {'name': 'Office SNR 25dB + Vol -6dB', 'noise_type': 'real', 'snr_db': 25, 'vol_reduction': 6, 'category': 'office'},
    {'name': 'Office SNR 25dB + Vol -9dB', 'noise_type': 'real', 'snr_db': 25, 'vol_reduction': 9, 'category': 'office'},
    {'name': 'Office SNR 25dB + Vol -12dB', 'noise_type': 'real', 'snr_db': 25, 'vol_reduction': 12, 'category': 'office'},
    
    # 38-41. Classroom noise with volume reduction (High SNR)
    {'name': 'Classroom SNR 25dB + Vol -3dB', 'noise_type': 'real', 'snr_db': 25, 'vol_reduction': 3, 'category': 'classroom'},
    {'name': 'Classroom SNR 25dB + Vol -6dB', 'noise_type': 'real', 'snr_db': 25, 'vol_reduction': 6, 'category': 'classroom'},
    {'name': 'Classroom SNR 25dB + Vol -9dB', 'noise_type': 'real', 'snr_db': 25, 'vol_reduction': 9, 'category': 'classroom'},
    {'name': 'Classroom SNR 25dB + Vol -12dB', 'noise_type': 'real', 'snr_db': 25, 'vol_reduction': 12, 'category': 'classroom'},
    
    # 42-45. White noise with volume reduction (High SNR)
    {'name': 'White SNR 25dB + Vol -3dB', 'noise_type': 'white', 'snr_db': 25, 'vol_reduction': 3, 'category': None},
    {'name': 'White SNR 25dB + Vol -6dB', 'noise_type': 'white', 'snr_db': 25, 'vol_reduction': 6, 'category': None},
    {'name': 'White SNR 25dB + Vol -9dB', 'noise_type': 'white', 'snr_db': 25, 'vol_reduction': 9, 'category': None},
    {'name': 'White SNR 25dB + Vol -12dB', 'noise_type': 'white', 'snr_db': 25, 'vol_reduction': 12, 'category': None},
    
    # 46-49. Mixed noise with volume reduction (High SNR)
    {'name': 'Mixed SNR 25dB + Vol -3dB', 'noise_type': 'mixed', 'snr_db': 25, 'vol_reduction': 3, 'category': None},
    {'name': 'Mixed SNR 25dB + Vol -6dB', 'noise_type': 'mixed', 'snr_db': 25, 'vol_reduction': 6, 'category': None},
    {'name': 'Mixed SNR 25dB + Vol -9dB', 'noise_type': 'mixed', 'snr_db': 25, 'vol_reduction': 9, 'category': None},
    {'name': 'Mixed SNR 25dB + Vol -12dB', 'noise_type': 'mixed', 'snr_db': 25, 'vol_reduction': 12, 'category': None},
]

print(f"✅ Created {len(stress_test_configs)} stress test configurations!")
print("\n📊 Configuration Breakdown:")
print(f"  • Clean baseline: 1")
print(f"  • High SNR variants (15-30dB): 24")
print(f"  • With volume reduction: 24")
print(f"  • Total test scenarios: {len(stress_test_configs)}")

## Create Stress Test Datasets

In [ ]:
class StressTestDataset(Dataset):
    """Dataset with specific noise augmentation applied"""
    def __init__(self, base_dataset, augmentation, mel_transform):
        super(StressTestDataset, self).__init__()
        self.base = base_dataset
        self.augmentation = augmentation
        self.mel_transform = mel_transform

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        waveform, label = self.base[idx]
        # Apply noise augmentation
        augmented_waveform = self.augmentation(waveform)
        # Convert to mel spectrogram
        transformed = self.mel_transform(augmented_waveform)
        return transformed, label

# Create mel spectrogram transform (no additional augmentation)
mel_transform_only = Compose([
    to_mel_spectrogram, 
    mel_spectrogram_to_numpy, 
    ToTensor()
])

# Create all stress test datasets
stress_test_datasets = {}
print("Creating stress test datasets...\n")

for i, config in enumerate(stress_test_configs):
    print(f"[{i+1}/{len(stress_test_configs)}] Creating: {config['name']}")
    
    # Create augmentation for this config
    augmentation = StressTestAugmentation(
        noise_library=noise_library,
        noise_type=config['noise_type'],
        snr_db=config['snr_db'],
        volume_reduction_db=config['vol_reduction'],
        category=config['category']
    )
    
    # Create dataset with this augmentation
    dataset_obj = StressTestDataset(
        base_dataset=init_test_set,
        augmentation=augmentation,
        mel_transform=mel_transform_only
    )
    
    stress_test_datasets[config['name']] = {
        'dataset': dataset_obj,
        'config': config
    }

print(f"\n✅ Successfully created {len(stress_test_datasets)} stress test datasets!")
print(f"   Each dataset has {len(init_test_set)} test samples")

## Run Stress Tests

In [ ]:
def evaluate_stress_test(model, dataset, config_name, device):
    """
    Evaluate model on a stress test dataset
    
    Returns:
        Dictionary with accuracy metrics and predictions
    """
    dataloader = torch.utils.data.DataLoader(
        dataset,
        batch_size=32,
        shuffle=False
    )
    
    model.eval()
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = torch.squeeze(labels).to(device)
            outputs = model(inputs)
            
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    accuracy = correct / total
    
    # Calculate per-class accuracy
    per_class_correct = defaultdict(int)
    per_class_total = defaultdict(int)
    
    for pred, label in zip(all_predictions, all_labels):
        per_class_total[label] += 1
        if pred == label:
            per_class_correct[label] += 1
    
    per_class_accuracy = {
        label: per_class_correct[label] / per_class_total[label]
        for label in per_class_total
    }
    
    return {
        'config_name': config_name,
        'accuracy': accuracy,
        'correct': correct,
        'total': total,
        'predictions': all_predictions,
        'labels': all_labels,
        'per_class_accuracy': per_class_accuracy
    }

In [ ]:
# Run all stress tests
print("="*80)
print("🔥 RUNNING COMPREHENSIVE STRESS TESTS")
print("="*80)
print(f"\nTesting model on {len(stress_test_datasets)} different noise configurations...")
print(f"Each test uses {len(init_test_set)} samples from the test set\n")

stress_test_results = []

for i, (name, test_data) in enumerate(stress_test_datasets.items()):
    print(f"\n[{i+1}/{len(stress_test_datasets)}] Testing: {name}")
    
    result = evaluate_stress_test(
        model=model,
        dataset=test_data['dataset'],
        config_name=name,
        device=device
    )
    
    stress_test_results.append(result)
    
    print(f"   ✓ Accuracy: {result['accuracy']*100:.2f}% ({result['correct']}/{result['total']})")

print("\n" + "="*80)
print("✅ ALL STRESS TESTS COMPLETED!")
print("="*80)

## Analyze Results

In [ ]:
# Create comprehensive results DataFrame
results_df = pd.DataFrame([
    {
        'Test_Name': r['config_name'],
        'Accuracy': r['accuracy'] * 100,
        'Correct': r['correct'],
        'Total': r['total']
    }
    for r in stress_test_results
])

# Sort by accuracy
results_df = results_df.sort_values('Accuracy', ascending=False).reset_index(drop=True)

# Display top and bottom performers
print("="*80)
print("📈 TOP 10 BEST PERFORMING CONFIGURATIONS")
print("="*80)
display(results_df.head(10))

print("\n" + "="*80)
print("📉 TOP 10 WORST PERFORMING CONFIGURATIONS")
print("="*80)
display(results_df.tail(10))

# Summary statistics
print("\n" + "="*80)
print("📊 SUMMARY STATISTICS")
print("="*80)
print(f"Best Accuracy:     {results_df['Accuracy'].max():.2f}%")
print(f"Worst Accuracy:    {results_df['Accuracy'].min():.2f}%")
print(f"Average Accuracy:  {results_df['Accuracy'].mean():.2f}%")
print(f"Median Accuracy:   {results_df['Accuracy'].median():.2f}%")
print(f"Std Dev:           {results_df['Accuracy'].std():.2f}%")
print("="*80)

In [ ]:
# Visualize stress test results
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 1. Bar plot of all accuracies
ax1 = axes[0, 0]
x_pos = np.arange(len(results_df))
bars = ax1.bar(x_pos, results_df['Accuracy'], alpha=0.8, color='steelblue')
# Color code: green for >90%, yellow for 70-90%, red for <70%
for i, (idx, row) in enumerate(results_df.iterrows()):
    if row['Accuracy'] >= 90:
        bars[i].set_color('green')
    elif row['Accuracy'] >= 70:
        bars[i].set_color('orange')
    else:
        bars[i].set_color('red')
ax1.set_xlabel('Test Configuration Index', fontsize=12)
ax1.set_ylabel('Accuracy (%)', fontsize=12)
ax1.set_title('Stress Test Results - All Configurations', fontsize=14, fontweight='bold')
ax1.axhline(y=results_df['Accuracy'].mean(), color='r', linestyle='--', label=f'Mean: {results_df["Accuracy"].mean():.2f}%')
ax1.axhline(y=90, color='g', linestyle=':', alpha=0.5, label='90% threshold')
ax1.axhline(y=70, color='orange', linestyle=':', alpha=0.5, label='70% threshold')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# 2. Grouped analysis by noise type
ax2 = axes[0, 1]
results_df['Noise_Type'] = results_df['Test_Name'].apply(lambda x: 
    'Clean' if 'Clean' in x else
    'City' if 'City' in x else
    'Office' if 'Office' in x else
    'Classroom' if 'Classroom' in x else
    'Common' if 'Common' in x else
    'White' if 'White' in x else
    'Gaussian' if 'Gaussian' in x else
    'Mixed' if 'Mixed' in x else 'Other'
)
noise_group = results_df.groupby('Noise_Type')['Accuracy'].agg(['mean', 'std', 'min', 'max'])
noise_group = noise_group.sort_values('mean', ascending=False)
x_pos_noise = np.arange(len(noise_group))
bars2 = ax2.bar(x_pos_noise, noise_group['mean'], yerr=noise_group['std'], 
                capsize=5, alpha=0.8, color='skyblue')
ax2.set_xticks(x_pos_noise)
ax2.set_xticklabels(noise_group.index, rotation=45, ha='right')
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Average Accuracy by Noise Type', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
# Add value labels on bars
for i, (idx, row) in enumerate(noise_group.iterrows()):
    ax2.text(i, row['mean'] + row['std'] + 1, f"{row['mean']:.1f}%", 
             ha='center', va='bottom', fontsize=10)

# 3. SNR level analysis
ax3 = axes[1, 0]
results_df['SNR'] = results_df['Test_Name'].apply(lambda x: 
    'Clean' if 'Clean' in x else
    '30dB' if '30dB' in x else
    '25dB' if '25dB' in x else
    '20dB' if '20dB' in x else
    '15dB' if '15dB' in x else 'Other'
)
snr_group = results_df.groupby('SNR')['Accuracy'].agg(['mean', 'std', 'count'])
snr_order = ['Clean', '30dB', '25dB', '20dB', '15dB']
snr_group = snr_group.reindex([s for s in snr_order if s in snr_group.index])
x_pos_snr = np.arange(len(snr_group))
bars3 = ax3.bar(x_pos_snr, snr_group['mean'], yerr=snr_group['std'], 
                capsize=5, alpha=0.8, color='lightcoral')
ax3.set_xticks(x_pos_snr)
ax3.set_xticklabels(snr_group.index, rotation=0)
ax3.set_ylabel('Accuracy (%)', fontsize=12)
ax3.set_xlabel('SNR Level', fontsize=12)
ax3.set_title('Average Accuracy by SNR Level', fontsize=14, fontweight='bold')
ax3.grid(axis='y', alpha=0.3)
# Add count and value labels
for i, (idx, row) in enumerate(snr_group.iterrows()):
    ax3.text(i, row['mean'] + row['std'] + 1, 
             f"{row['mean']:.1f}%\n(n={int(row['count'])})", 
             ha='center', va='bottom', fontsize=10)

# 4. Volume reduction analysis
ax4 = axes[1, 1]
results_df['Vol_Reduction'] = results_df['Test_Name'].apply(lambda x: 
    'None' if 'Vol' not in x else
    '-3dB' if '-3dB' in x else
    '-6dB' if '-6dB' in x else
    '-9dB' if '-9dB' in x else
    '-12dB' if '-12dB' in x else 'Other'
)
vol_group = results_df.groupby('Vol_Reduction')['Accuracy'].agg(['mean', 'std', 'count'])
vol_order = ['None', '-3dB', '-6dB', '-9dB', '-12dB']
vol_group = vol_group.reindex([v for v in vol_order if v in vol_group.index])
x_pos_vol = np.arange(len(vol_group))
bars4 = ax4.bar(x_pos_vol, vol_group['mean'], yerr=vol_group['std'], 
                capsize=5, alpha=0.8, color='lightgreen')
ax4.set_xticks(x_pos_vol)
ax4.set_xticklabels(vol_group.index, rotation=0)
ax4.set_ylabel('Accuracy (%)', fontsize=12)
ax4.set_xlabel('Volume Reduction', fontsize=12)
ax4.set_title('Average Accuracy by Volume Reduction', fontsize=14, fontweight='bold')
ax4.grid(axis='y', alpha=0.3)
# Add count and value labels
for i, (idx, row) in enumerate(vol_group.iterrows()):
    ax4.text(i, row['mean'] + row['std'] + 1, 
             f"{row['mean']:.1f}%\n(n={int(row['count'])})", 
             ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('stress_test_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Visualization saved as 'stress_test_analysis.png'")

In [ ]:
# Export detailed results to CSV
results_df.to_csv('stress_test_results_detailed.csv', index=False)
print("✅ Detailed results exported to 'stress_test_results_detailed.csv'")

# Create summary table by noise type and SNR
summary_pivot = results_df.pivot_table(
    values='Accuracy',
    index='Noise_Type',
    columns='SNR',
    aggfunc='mean'
)

print("\n" + "="*80)
print("📊 ACCURACY HEATMAP: Noise Type vs SNR Level")
print("="*80)
display(summary_pivot.round(2))

# Visualize as heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(summary_pivot, annot=True, fmt='.2f', cmap='RdYlGn', 
            vmin=60, vmax=100, cbar_kws={'label': 'Accuracy (%)'})
plt.title('Model Accuracy Heatmap: Noise Type vs SNR Level', fontsize=14, fontweight='bold')
plt.xlabel('SNR Level', fontsize=12)
plt.ylabel('Noise Type', fontsize=12)
plt.tight_layout()
plt.savefig('stress_test_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Heatmap saved as 'stress_test_heatmap.png'")

## Per-Class Performance Analysis

In [ ]:
# Analyze per-class performance across all stress tests
class_performance = defaultdict(list)

for result in stress_test_results:
    per_class = result['per_class_accuracy']
    for class_id, accuracy in per_class.items():
        class_performance[class_id].append(accuracy)

# Calculate statistics for each class
class_stats = {}
for class_id, accuracies in class_performance.items():
    class_stats[class_id] = {
        'character': label_dict[class_id],
        'mean': np.mean(accuracies) * 100,
        'std': np.std(accuracies) * 100,
        'min': np.min(accuracies) * 100,
        'max': np.max(accuracies) * 100,
        'median': np.median(accuracies) * 100
    }

# Create DataFrame
class_stats_df = pd.DataFrame(class_stats).T
class_stats_df = class_stats_df.sort_values('mean', ascending=False)

print("="*80)
print("📊 PER-CLASS PERFORMANCE ACROSS ALL STRESS TESTS")
print("="*80)
print("\nTop 10 Most Robust Characters:")
display(class_stats_df.head(10))

print("\nBottom 10 Most Vulnerable Characters:")
display(class_stats_df.tail(10))

# Visualize per-class performance
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Plot 1: Mean accuracy per character with error bars
ax1 = axes[0]
x_pos = np.arange(len(class_stats_df))
bars = ax1.bar(x_pos, class_stats_df['mean'], yerr=class_stats_df['std'], 
               capsize=3, alpha=0.7, color='steelblue')
ax1.set_xticks(x_pos[::2])  # Show every other tick to avoid crowding
ax1.set_xticklabels(class_stats_df['character'].values[::2], rotation=0, fontsize=10)
ax1.set_ylabel('Mean Accuracy (%)', fontsize=12)
ax1.set_xlabel('Character', fontsize=12)
ax1.set_title('Per-Character Performance Across All Stress Tests', fontsize=14, fontweight='bold')
ax1.axhline(y=class_stats_df['mean'].mean(), color='r', linestyle='--', 
            label=f'Overall Mean: {class_stats_df["mean"].mean():.2f}%')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Plot 2: Box plot showing distribution
ax2 = axes[1]
# Get top 15 and bottom 15 characters
top_chars = class_stats_df.head(15).index
bottom_chars = class_stats_df.tail(15).index
selected_chars = list(top_chars) + list(bottom_chars)

data_for_boxplot = []
labels_for_boxplot = []
for class_id in selected_chars:
    if class_id in class_performance:
        data_for_boxplot.append([acc * 100 for acc in class_performance[class_id]])
        labels_for_boxplot.append(label_dict[class_id])

bp = ax2.boxplot(data_for_boxplot, labels=labels_for_boxplot, patch_artist=True)
# Color top performers in green, bottom in red
for i, patch in enumerate(bp['boxes']):
    if i < 15:
        patch.set_facecolor('lightgreen')
    else:
        patch.set_facecolor('lightcoral')
ax2.set_xticklabels(labels_for_boxplot, rotation=45, ha='right', fontsize=9)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_xlabel('Character (Green=Top 15, Red=Bottom 15)', fontsize=12)
ax2.set_title('Accuracy Distribution: Best vs Worst Characters', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('per_class_stress_test_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Per-class analysis saved as 'per_class_stress_test_analysis.png'")

## Comprehensive Report

In [ ]:
# Generate comprehensive stress test report
print("="*80)
print("📋 COMPREHENSIVE STRESS TEST REPORT")
print("="*80)
print("\n")
print("🔥 TEST SCOPE")
print("-" * 80)
print(f"  Total configurations tested:         {len(stress_test_results)}")
print(f"  Samples per configuration:           {len(init_test_set)}")
print(f"  Total predictions made:              {len(stress_test_results) * len(init_test_set)}")
print(f"  Number of classes:                   {len(label_dict)}")
print("\n")

print("📊 OVERALL PERFORMANCE")
print("-" * 80)
print(f"  Best accuracy:                       {results_df['Accuracy'].max():.2f}%")
print(f"  Best configuration:                  {results_df.iloc[0]['Test_Name']}")
print(f"  Worst accuracy:                      {results_df['Accuracy'].min():.2f}%")
print(f"  Worst configuration:                 {results_df.iloc[-1]['Test_Name']}")
print(f"  Average accuracy:                    {results_df['Accuracy'].mean():.2f}%")
print(f"  Median accuracy:                     {results_df['Accuracy'].median():.2f}%")
print(f"  Standard deviation:                  {results_df['Accuracy'].std():.2f}%")
print(f"  Configurations with >90% accuracy:   {len(results_df[results_df['Accuracy'] >= 90])}")
print(f"  Configurations with >80% accuracy:   {len(results_df[results_df['Accuracy'] >= 80])}")
print(f"  Configurations with >70% accuracy:   {len(results_df[results_df['Accuracy'] >= 70])}")
print("\n")

print("🎵 NOISE TYPE PERFORMANCE RANKING")
print("-" * 80)
for i, (noise_type, stats) in enumerate(noise_group.iterrows(), 1):
    print(f"  {i}. {noise_type:15s} - Mean: {stats['mean']:6.2f}% | "
          f"Std: {stats['std']:5.2f}% | Range: [{stats['min']:6.2f}% - {stats['max']:6.2f}%]")
print("\n")

print("📡 SNR LEVEL IMPACT")
print("-" * 80)
for snr_level, stats in snr_group.iterrows():
    print(f"  {snr_level:8s} (n={int(stats['count']):2d}) - "
          f"Mean: {stats['mean']:6.2f}% | Std: {stats['std']:5.2f}%")
print("\n")

print("🔉 VOLUME REDUCTION IMPACT")
print("-" * 80)
for vol_level, stats in vol_group.iterrows():
    print(f"  {vol_level:8s} (n={int(stats['count']):2d}) - "
          f"Mean: {stats['mean']:6.2f}% | Std: {stats['std']:5.2f}%")
print("\n")

print("🎯 CHARACTER-LEVEL ROBUSTNESS")
print("-" * 80)
print(f"  Most robust character:               {class_stats_df.iloc[0]['character']} "
      f"({class_stats_df.iloc[0]['mean']:.2f}% mean accuracy)")
print(f"  Most vulnerable character:           {class_stats_df.iloc[-1]['character']} "
      f"({class_stats_df.iloc[-1]['mean']:.2f}% mean accuracy)")
print(f"  Average per-class accuracy:          {class_stats_df['mean'].mean():.2f}%")
print(f"  Characters with >90% mean accuracy:  {len(class_stats_df[class_stats_df['mean'] >= 90])}")
print(f"  Characters with >80% mean accuracy:  {len(class_stats_df[class_stats_df['mean'] >= 80])}")
print(f"  Characters with >70% mean accuracy:  {len(class_stats_df[class_stats_df['mean'] >= 70])}")
print("\n")

print("💡 KEY FINDINGS")
print("-" * 80)
# Identify best and worst performing noise types
best_noise = noise_group.idxmax()['mean']
worst_noise = noise_group.idxmin()['mean']
print(f"  • Model performs best with: {best_noise} noise")
print(f"  • Model struggles most with: {worst_noise} noise")

# SNR impact
if 'Clean' in snr_group.index and '15dB' in snr_group.index:
    snr_drop = snr_group.loc['Clean', 'mean'] - snr_group.loc['15dB', 'mean']
    print(f"  • Accuracy drop from Clean to 15dB SNR: {snr_drop:.2f}%")

# Volume reduction impact
if 'None' in vol_group.index and '-12dB' in vol_group.index:
    vol_drop = vol_group.loc['None', 'mean'] - vol_group.loc['-12dB', 'mean']
    print(f"  • Accuracy drop from no reduction to -12dB: {vol_drop:.2f}%")

# Character robustness spread
char_range = class_stats_df['mean'].max() - class_stats_df['mean'].min()
print(f"  • Character performance spread: {char_range:.2f}%")

print("\n")
print("="*80)
print("✅ STRESS TEST REPORT COMPLETE")
print("="*80)
print("\n📁 Generated Files:")
print("  • stress_test_results_detailed.csv")
print("  • stress_test_analysis.png")
print("  • stress_test_heatmap.png")
print("  • per_class_stress_test_analysis.png")
print("="*80)